# InftyThink: Iterative Reasoning with Bounded Memory on Kaggle T4 GPU

This notebook runs the full InftyThink training and experiment pipeline on a Kaggle free T4 GPU.

**InftyThink** is a JAX/Flax ~35M-parameter, 6-layer causal transformer that explores iterative
chain-of-thought reasoning with bounded context. The model processes long reasoning traces in
fixed-length segments, compressing each segment into a summary that conditions the next step.
This decouples reasoning depth from context length.

**Pipeline overview:**
1. Clone code from GitHub and install dependencies
2. Prepare data from `open-r1/OpenR1-Math-220k`
3. Train the model (5,000 steps, ~2-3 hrs on T4)
4. Run key experiments: baselines, InftyThink, structured state
5. Analyze results and generate figures

**Prerequisites:**
- Enable **GPU T4 x2** (or T4 x1) accelerator in notebook settings
- Enable **Internet** access in notebook settings
- Estimated total runtime: ~3-4 hours

In [ ]:
# =============================================================================
# Cell 2: Clone code from GitHub
# =============================================================================
# Clone the InftyThink repo directly — no dataset upload needed.
# Just requires Internet to be enabled in notebook settings.
# =============================================================================

import os

DST = "/kaggle/working/reasoning"

if os.path.exists(DST):
    print(f"Code already exists at {DST}, skipping clone.")
else:
    !git clone https://github.com/jadenfix/InftyThink.git {DST}

os.chdir(DST)
print(f"Working directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
%%time
# =============================================================================
# Cell 3: Install dependencies and verify GPU
# =============================================================================

import subprocess, sys

# Kaggle T4 comes with CUDA 12.x pre-installed.
# Install JAX with CUDA support and all other deps.
!pip install -q --upgrade pip
!pip install -q -r requirements-gpu.txt

# Verify JAX sees the GPU
import jax
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"GPU available: {any('gpu' in str(d).lower() for d in jax.devices())}")

# Quick sanity check
import jax.numpy as jnp
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
print(f"Matrix multiply test (1000x1000): sum = {float(y.sum()):.0f} (expected 1000000)")

assert any('gpu' in str(d).lower() for d in jax.devices()), \
    "No GPU detected! Enable GPU in Kaggle notebook settings."

In [ ]:
%%time
# =============================================================================
# Cell 4: Prepare data
# =============================================================================
# Downloads open-r1/OpenR1-Math-220k from HuggingFace and computes
# dataset statistics (token length histograms, segments per example, etc.)

import os
os.chdir("/kaggle/working/reasoning")

!python main.py prepare-data --n-train 5000 --n-eval 500

In [ ]:
%%time
# =============================================================================
# Cell 5: Train the model
# =============================================================================
# ~35M param, 6-layer transformer. Reduced from 10K to 5K steps to fit
# within Kaggle's T4 session time (~2-3 hours for training).
#
# Config: batch_size=8, grad_accum=4 (effective batch=32), lr=3e-4
# We patch max_steps via a temporary config override.

import os, yaml
os.chdir("/kaggle/working/reasoning")

# Create a Kaggle-specific config with reduced steps
with open("configs/base.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["training"]["max_steps"] = 5000
cfg["training"]["eval_every"] = 500
cfg["training"]["save_every"] = 1000

os.makedirs("configs", exist_ok=True)
with open("configs/kaggle.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Training config (kaggle.yaml):")
print(yaml.dump(cfg, default_flow_style=False))

!python main.py train --config configs/kaggle.yaml

In [ ]:
%%time
# =============================================================================
# Cell 6: Run key experiments
# =============================================================================
# We run the three most important experiments rather than all 10 to stay
# within the Kaggle time budget:
#   B1 - Vanilla CoT (baseline)
#   M1 - InftyThink  (main method)
#   E1 - Structured State (extension)
#
# Each experiment loads the trained checkpoint, runs inference on the eval
# set, and saves metrics to results/.

import os
os.chdir("/kaggle/working/reasoning")

experiments = [
    ("b1_vanilla_cot",    "Baseline B1: Vanilla CoT"),
    ("m1_inftythink",     "Main Method M1: InftyThink"),
    ("e1_structured_state", "Extension E1: Structured State"),
]

for exp_name, label in experiments:
    print(f"\n{'='*60}")
    print(f"  Running: {label}")
    print(f"{'='*60}")
    exit_code = os.system(f"python main.py run-experiment --name {exp_name}")
    if exit_code != 0:
        print(f"WARNING: {exp_name} exited with code {exit_code}")
    else:
        print(f"  {label} completed successfully.")

# List result files
print("\nResult files:")
for root, dirs, files in os.walk("results"):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path} ({size:,} bytes)")

In [ ]:
%%time
# =============================================================================
# Cell 7: Run analysis and generate figures
# =============================================================================
# Generates up to 9 publication-quality figures:
#   fig1 - Accuracy by method (bar + CI)
#   fig2 - Token efficiency
#   fig3 - Peak context length
#   fig8 - Failure breakdown
#   fig9 - Structured vs free-form
# (Ablation figures require running all 10 experiments)

import os
os.chdir("/kaggle/working/reasoning")

!python main.py analyze --results-dir results/

# List generated figures
figures_dir = "results/figures"
if os.path.isdir(figures_dir):
    figs = sorted(os.listdir(figures_dir))
    print(f"\nGenerated {len(figs)} figures:")
    for fig in figs:
        print(f"  {fig}")
else:
    print("No figures directory found.")

In [ ]:
# =============================================================================
# Cell 8: Display generated figures inline
# =============================================================================

import os
import glob
from IPython.display import display, Image, Markdown

os.chdir("/kaggle/working/reasoning")

figures_dir = "results/figures"
fig_files = sorted(glob.glob(os.path.join(figures_dir, "*.png")))

if not fig_files:
    print("No figures found. Check that the experiments and analysis ran successfully.")
else:
    for fig_path in fig_files:
        fig_name = os.path.basename(fig_path).replace(".png", "").replace("_", " ").title()
        display(Markdown(f"### {fig_name}"))
        display(Image(filename=fig_path, width=700))
        print()

In [ ]:
# =============================================================================
# Cell 9: Save and download results
# =============================================================================
# Package everything into a zip for easy download from Kaggle.

import os
import shutil
import json

os.chdir("/kaggle/working/reasoning")

# Print a summary of key metrics
print("=" * 60)
print("  RESULTS SUMMARY")
print("=" * 60)

result_files = [
    ("results/b1_vanilla_cot.json",    "B1 Vanilla CoT"),
    ("results/m1_inftythink.json",     "M1 InftyThink"),
    ("results/e1_structured_state.json", "E1 Structured State"),
]

for path, label in result_files:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        acc = data.get("accuracy", "N/A")
        eff = data.get("token_efficiency", "N/A")
        if isinstance(acc, float):
            acc = f"{acc:.4f}"
        if isinstance(eff, float):
            eff = f"{eff:.1f}"
        print(f"  {label:25s}  accuracy={acc}  token_eff={eff}")
    else:
        print(f"  {label:25s}  (not found)")

print()

# Create a zip archive of all results + checkpoints
output_zip = "/kaggle/working/inftythink_results"

# Collect everything worth saving
items_to_zip = []
for d in ["results", "checkpoints"]:
    if os.path.isdir(d):
        items_to_zip.append(d)
# Also include the Kaggle config for reproducibility
if os.path.exists("configs/kaggle.yaml"):
    items_to_zip.append("configs/kaggle.yaml")

if items_to_zip:
    # Create archive
    shutil.make_archive(output_zip, "zip", ".", ".")
    # More targeted zip: just results and checkpoints
    import zipfile
    zip_path = output_zip + ".zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in items_to_zip:
            if os.path.isdir(item):
                for root, dirs, files in os.walk(item):
                    for file in files:
                        filepath = os.path.join(root, file)
                        zf.write(filepath)
            else:
                zf.write(item)

    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Results archive: {zip_path} ({size_mb:.1f} MB)")
    print("Download from the Output tab on the right, or use:")
    print("  from IPython.display import FileLink")
    print("  FileLink(f'{zip_path}')")
else:
    print("No results or checkpoints found to archive.")

print("\nDone! Pipeline complete.")